# Optimizers: Parameter iteration and iterator exhaustion

**Solution notebook — Delta Drills #445**

Run the cells top-to-bottom to see the reference answer execute.


## Problem

An optimizer touches every parameter on every step, so the parameter iterator must be materialized ONCE, up front. Write solve(model, lr, steps) for a module whose parameters already carry gradients (`p.grad` is set): capture `params = list(model.parameters())` once, then run `steps` SGD steps — on each step, inside `torch.no_grad()`, walk `params` and move each parameter one step against its gradient, in place (`p -= lr * p.grad`). Return the model's weight tensor (detached is fine).


<details><summary>💡 Hint (click to reveal)</summary>

`params = list(model.parameters())` once; each step `p -= lr * p.grad` under no_grad.

</details>


In [ ]:
%pip install -q numpy torch --index-url https://download.pytorch.org/whl/cpu

## Reference solution


In [ ]:
import torch
import torch.nn as nn

def solve(model, lr, steps):
    params = list(model.parameters())
    for _ in range(steps):
        with torch.no_grad():
            for p in params:
                p -= lr * p.grad
    return model.weight.detach()

model = nn.Linear(3, 2, bias=False)
with torch.no_grad():
    model.weight.copy_(torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]))
model.weight.grad = torch.ones_like(model.weight)
print(solve(model, 0.1, 3))


## Why this works

The optimizer pattern in miniature: materialize the parameter list ONCE (a generator would be empty on step 2), then per step walk it under `torch.no_grad()` applying `p -= lr * p.grad` in place. Three steps of lr·1.0 gradients move every weight down by exactly 0.3.
